In [ ]:

import pandas as pd

df = pd.read_csv('BDD_Billboard_copia_Limpia_2000.csv')
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['date'] = df['date'].dt.year

df = df.drop(columns=['image_url', 'rank', 'weeks_in_charts', 'last_week'])

# Renombrar columnas
df = df.rename(columns={
    'date': 'Año',
    'song': 'Canción',
    'artist': 'Artista',
    'peak_position': 'Máxima_posición'
})

# Columna 'Canción' ya renombrada
df = df.drop(columns=['Canción'])
df_filtered = df[(df['Máxima_posición'] >= 1) & (df['Máxima_posición'] <= 10)]
df_small = df_filtered[df_filtered['Año'].between(2000, 2010)]


df.head()




,Año,Artista,Máxima_posición
0,2000,Santana Featuring Rob Thomas,1
1,2000,Brian McKnight,2
2,2000,Jessica Simpson,3
3,2000,Whitney Houston,4
4,2000,"Missy ""Misdemeanor"" Elliott Featuring NAS| EVE...",5


In [ ]:
import pandas as pd
import altair as alt
from google.colab import files  # Para subir archivos si no existe el CSV

# --- Cargar archivo ---
try:
    df = pd.read_csv('BDD_Billboard_copia_Limpia_2000.csv')
except FileNotFoundError:
    print("Error: El archivo 'BDD_Billboard_copia_Limpia_2000.csv' no fue encontrado.")
    print("Por favor, súbelo con:")
    print("from google.colab import files; uploaded = files.upload()")
    raise

# --- Preparar datos ---
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['date'] = df['date'].dt.year

df = df.drop(columns=['image_url', 'rank', 'weeks_in_charts', 'last_week'])

df = df.rename(columns={
    'date': 'Año',
    'song': 'Canción',
    'artist': 'Artista',
    'peak_position': 'Máxima_posición'
})

df = df.drop(columns=['Canción'])
df_filtered = df[(df['Máxima_posición'] >= 1) & (df['Máxima_posición'] <= 10)]
df_small = df_filtered[df_filtered['Año'].between(2000, 2010)]

pivot_table = df_small.pivot_table(
    index='Artista',
    columns='Año',
    values='Máxima_posición',
    aggfunc='count',
    fill_value=0
)

# Top 20 artistas con más canciones en el top 10
top_artistas = pivot_table.sum(axis=1).sort_values(ascending=False).head(20).index
pivot_table_top = pivot_table.loc[top_artistas]

# --- Transformar datos a formato largo ---
df_long = pivot_table_top.reset_index().melt(
    id_vars='Artista',
    var_name='Año',
    value_name='Cantidad'
)

df_long['Año'] = df_long['Año'].astype(str)

# --- Crear selecciones interactivas ---
brush = alt.selection_interval(encodings=['x', 'y'])
highlight = alt.selection_point(fields=['Artista'], empty='none')

# --- Heatmap en tonos morados ---
heatmap = (
    alt.Chart(df_long)
    .mark_rect()
    .encode(
        x=alt.X('Año:O', title='Año'),
        y=alt.Y(
            'Artista:O',
            title='Artista',
            sort=alt.EncodingSortField(field='Cantidad', order='descending')
        ),
        color=alt.condition(
            alt.datum.Cantidad > 0,
            alt.Color('Cantidad:Q',
                      scale=alt.Scale(scheme='purples'),
                      title='Cantidad de canciones'),
            alt.value('white')  # Deja en blanco los espacios sin valor
        ),
        tooltip=['Artista', 'Año', 'Cantidad']
    )
    .properties(
        width=700,
        height=500,
        title='Cantidad de canciones en top 10 por artista y año (2000–2010)'
    )
    .add_params(brush, highlight)
    .transform_filter(brush)
)

# --- Texto dinámico al resaltar artista ---
text = (
    alt.Chart(df_long)
    .mark_text(
        align='left',
        baseline='middle',
        dx=5,
        dy=-5,
        fontWeight='bold',
        color='black'
    )
    .encode(
        x='Año:O',
        y='Artista:O',
        text=alt.condition(highlight, 'Cantidad:Q', alt.value(''))
    )
    .add_params(highlight)
)

# --- Combinar y aplicar estilo con Poppins ---
chart = (heatmap + text).configure(
    title=alt.TitleConfig(font='Poppins', fontSize=18, anchor='start', color='#4527a0'),
    axis=alt.AxisConfig(labelFont='Poppins', titleFont='Poppins', labelFontSize=12, titleFontSize=13),
    legend=alt.LegendConfig(labelFont='Poppins', titleFont='Poppins'),
    view=alt.ViewConfig(stroke='transparent'),  # sin borde
    background='white'  # fondo blanco limpio
)

chart


alt.LayerChart(...)

In [ ]:
import pandas as pd


df = pd.read_csv('BDD_Billboard_copia_Limpia_2000.csv')

# Convertir columna 'date' a datetime y extraer solo el año
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['date'] = df['date'].dt.year

# Eliminar columnas
df = df.drop(columns=['image_url', 'rank', 'weeks_in_charts', 'last_week'])


df = df.rename(columns={
    'date': 'Año',
    'song': 'Canción',
    'artist': 'Artista',
    'peak_position': 'Máxima_posición'
})

# Eliminar la columna 'Canción'
df = df.drop(columns=['Canción'])

# Filtrar para máximo posición entre 1 y 10
df_filtered = df[(df['Máxima_posición'] >= 1) & (df['Máxima_posición'] <= 10)]

# Dividir en dos dataframes por rangos de años
df_2000_2010 = df_filtered[df_filtered['Año'].between(2000, 2010)]
df_2011_2025 = df_filtered[df_filtered['Año'].between(2011, 2025)]


print("\nDatos 2011-2025:")
print(df_2011_2025.head())


Datos 2011-2025:
        Año                  Artista  Máxima_posición
57400  2011               Bruno Mars                1
57401  2011               Katy Perry                1
57402  2011                    Ke$ha                1
57403  2011  Rihanna Featuring Drake                1
57404  2011                     P!nk                1


In [ ]:
import pandas as pd


df = pd.read_csv('BDD_Billboard_copia_Limpia_2000.csv')

# Convertir columna 'date' a datetime y extraer solo el año
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['date'] = df['date'].dt.year

# Eliminar columnas
df = df.drop(columns=['image_url', 'rank', 'weeks_in_charts', 'last_week'])


df = df.rename(columns={
    'date': 'Año',
    'song': 'Canción',
    'artist': 'Artista',
    'peak_position': 'Máxima_posición'
})

# Eliminar la columna 'Canción'
df = df.drop(columns=['Canción'])

# Filtrar para máximo posición entre 1 y 10
df_filtered = df[(df['Máxima_posición'] >= 1) & (df['Máxima_posición'] <= 10)]


df_2011_2025 = df_filtered[df_filtered['Año'].between(2011, 2025)]


print("\nDatos 2011-2025:")
print(df_2011_2025.head())


Datos 2011-2025:
        Año                  Artista  Máxima_posición
57400  2011               Bruno Mars                1
57401  2011               Katy Perry                1
57402  2011                    Ke$ha                1
57403  2011  Rihanna Featuring Drake                1
57404  2011                     P!nk                1


In [ ]:
import pandas as pd
import altair as alt

# ✨ Desactivar límites y vegafusion
alt.data_transformers.disable_max_rows()
alt.data_transformers.enable('default')

# === 1. CARGAR Y PROCESAR DATOS ===
df = pd.read_csv('BDD_Billboard_copia_Limpia_2000.csv')

# Convertir columna 'date' a año
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['Año'] = df['date'].dt.year

# Eliminar columnas innecesarias
df = df.drop(columns=['image_url', 'rank', 'weeks_in_charts', 'last_week'])

# Renombrar columnas
df = df.rename(columns={
    'song': 'Canción',
    'artist': 'Artista',
    'peak_position': 'Máxima_posición'
})

# Eliminar columna 'Canción'
df = df.drop(columns=['Canción'])

# Filtrar canciones con posición máxima entre 1 y 10
df_filtered = df[(df['Máxima_posición'] >= 1) & (df['Máxima_posición'] <= 10)]

# Filtrar rango de años 2011–2025
df_2011_2025 = df_filtered[df_filtered['Año'].between(2011, 2025)]

# === 2. AGRUPAR DATOS ===
pivot_table_top = (
    df_2011_2025
    .groupby(['Artista', 'Año'])
    .size()
    .unstack(fill_value=0)
)

# === 3. TRANSFORMAR DATOS PARA ALTAIR ===
df_long = (
    pivot_table_top
    .reset_index()
    .melt(id_vars='Artista', var_name='Año', value_name='Cantidad')
)

df_long['Año'] = df_long['Año'].astype(str)

# Mostrar solo los 20 artistas más frecuentes
top_artistas = (
    df_long.groupby('Artista')['Cantidad']
    .sum()
    .nlargest(20)
    .index
)
df_long = df_long[df_long['Artista'].isin(top_artistas)]

# === 4. CREAR VISUALIZACIÓN ===

# Selección para zoom
brush = alt.selection_interval(encodings=['x', 'y'])

# Selección para resaltar artista
highlight = alt.selection_point(fields=['Artista'], empty='none')

# --- Heatmap rosa con valores cero en blanco ---
heatmap = (
    alt.Chart(df_long)
    .mark_rect()
    .encode(
        x=alt.X('Año:O', title='Año'),
        y=alt.Y(
            'Artista:O',
            title='Artista',
            sort=alt.EncodingSortField(field='Cantidad', order='descending')
        ),
        color=alt.condition(
            alt.datum.Cantidad > 0,
            alt.Color('Cantidad:Q',
                      scale=alt.Scale(range=['#fde0dd', '#fa9fb5', '#c51b8a']),
                      title='Cantidad de canciones'),
            alt.value('white')  # ← Celdas sin valor quedan blancas
        ),
        tooltip=['Artista', 'Año', 'Cantidad']
    )
    .properties(
        width=700,
        height=500,
        title='Cantidad de entradas en Top 10 por artista y año (2011 - 2025)'
    )
    .add_params(brush, highlight)
    .transform_filter(brush)
)

# --- Texto al hacer clic ---
text = (
    alt.Chart(df_long)
    .mark_text(
        align='left',
        baseline='middle',
        dx=5,
        dy=-5,
        fontWeight='bold',
        color='black'
    )
    .encode(
        x='Año:O',
        y='Artista:O',
        text=alt.condition(highlight, 'Cantidad:Q', alt.value(''))
    )
    .add_params(highlight)
)

# --- Combinar ---
chart = (heatmap + text).configure(
    title=alt.TitleConfig(font='Poppins', fontSize=18, anchor='start', color='#c51b8a'),
    axis=alt.AxisConfig(labelFont='Poppins', titleFont='Poppins', labelFontSize=12, titleFontSize=13),
    legend=alt.LegendConfig(labelFont='Poppins', titleFont='Poppins'),
    view=alt.ViewConfig(stroke='transparent'),  # sin borde
    background='white'  # fondo blanco limpio
)

chart

alt.LayerChart(...)

In [ ]:
!jupyter nbconvert /content/codigo_visualizacion.ipynb --to html --output /content/codigo_visualizacion.html

[NbConvertApp] Converting notebook /content/codigo_visualizacion.ipynb to html
[NbConvertApp] Writing 369676 bytes to /content/codigo_visualizacion.html


In [ ]:
import pandas as pd

# --- Cargar archivo ---
df = pd.read_csv('Album_del_Año.csv')

df.head()


,A�o,Categor�a,Artista,Disco o canci�n,Ganador,G�nero del arista/banda,G�nero disco
0,2024,Album Of The Year,Charli xcx,BRAT,False,M,Dance Pop
1,2024,Album Of The Year,Jacob Collier,Djesse Vol. 4,False,H,Jazz
2,2024,Album Of The Year,Billie Eilish,HIT ME HARD AND SOFT,False,M,Pop
3,2024,Album Of The Year,Chappell Roan,The Rise And Fall Of A Midwest Princess,False,M,Pop
4,2024,Album Of The Year,Taylor Swift,THE TORTURED POETS DEPARTMENT,False,M,Pop


from matplotlib import pyplot as plt
import seaborn as sns
_df_0.groupby('Artista').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
_df_1.groupby('Disco o canci�n').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
_df_2.groupby('G�nero del arista/banda').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
_df_3.groupby('G�nero disco').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

...

from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  counted = (series['A�o']
                .value_counts()
              .reset_index(name='counts')
              .rename({'index': 'A�o'}, axis=1)
              .sort_values('A�o', ascending=True))
  xs = counted['A�o']
  ys = counted['counts']
  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_6.sort_values('A�o', ascending=True)
for i, (series_name, series) in enumerate(df_sorted.groupby('G�nero del arista/banda')):
  _plot_series(series, series_name, i)
  fig.legend(title='G�nero del arista/banda', bbox_to_anchor=(1, 1), loc='upper left')
sns.despine(fig=fig, ax=ax)
plt.xlabel('A�o')
_ = plt.ylabel('count()')

from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  counted = (series['A�o']
                .value_counts()
              .reset_index(name='counts')
              .rename({'index': 'A�o'}, axis=1)
              .sort_values('A�o', ascending=True))
  xs = counted['A�o']
  ys = counted['counts']
  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_7.sort_values('A�o', ascending=True)
for i, (series_name, series) in enumerate(df_sorted.groupby('G�nero disco')):
  _plot_series(series, series_name, i)
  fig.legend(title='G�nero disco', bbox_to_anchor=(1, 1), loc='upper left')
sns.despine(fig=fig, ax=ax)
plt.xlabel('A�o')
_ = plt.ylabel('count()')

from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
plt.subplots(figsize=(8, 8))
df_2dhist = pd.DataFrame({
    x_label: grp['Disco o canci�n'].value_counts()
    for x_label, grp in _df_8.groupby('Artista')
})
sns.heatmap(df_2dhist, cmap='viridis')
plt.xlabel('Artista')
_ = plt.ylabel('Disco o canci�n')

from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
plt.subplots(figsize=(8, 8))
df_2dhist = pd.DataFrame({
    x_label: grp['G�nero del arista/banda'].value_counts()
    for x_label, grp in _df_9.groupby('Disco o canci�n')
})
sns.heatmap(df_2dhist, cmap='viridis')
plt.xlabel('Disco o canci�n')
_ = plt.ylabel('G�nero del arista/banda')

from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
plt.subplots(figsize=(8, 8))
df_2dhist = pd.DataFrame({
    x_label: grp['G�nero disco'].value_counts()
    for x_label, grp in _df_10.groupby('G�nero del arista/banda')
})
sns.heatmap(df_2dhist, cmap='viridis')
plt.xlabel('G�nero del arista/banda')
_ = plt.ylabel('G�nero disco')

In [ ]:
import pandas as pd


grammys = pd.read_csv("Album_del_Ano.csv", encoding="latin1")


grammys = grammys.rename(columns={
    'A�o': 'Año',
    'Categor�a': 'Categoría',
    'Disco o canci�n': 'Disco o canción',
    'Ganador': 'Ganador',
    'G�nero del arista/banda': 'Género del artista/banda',
    'G�nero disco': 'Género disco'
})

grammys.head()

,Aï¿½o,Categorï¿½a,Artista,Disco o canciï¿½n,Ganador,Gï¿½nero del arista/banda,Gï¿½nero disco
0,2024,Album Of The Year,Charli xcx,BRAT,False,M,Dance Pop
1,2024,Album Of The Year,Jacob Collier,Djesse Vol. 4,False,H,Jazz
2,2024,Album Of The Year,Billie Eilish,HIT ME HARD AND SOFT,False,M,Pop
3,2024,Album Of The Year,Chappell Roan,The Rise And Fall Of A Midwest Princess,False,M,Pop
4,2024,Album Of The Year,Taylor Swift,THE TORTURED POETS DEPARTMENT,False,M,Pop


In [ ]:
import pandas as pd


billboard = pd.read_csv("BBD_Billboard_copia_Limpieza_2000.csv")
grammys = pd.read_csv("Album_del_Ano.csv")


grammys = grammys.rename(columns={
    'A�o': 'Año',
    'Categor�a': 'Categoría',
    'Disco o canci�n': 'Disco o canción',
    'G�nero del arista/banda': 'Género del artista/banda',
    'G�nero disco': 'Género disco'
})


print(grammys.columns)
grammys.head()

Index(['Año', 'Categoría', 'Artista', 'Disco o canción', 'Ganador',
       'Género del artista/banda', 'Género disco'],
      dtype='object')


,Año,Categoría,Artista,Disco o canción,Ganador,Género del artista/banda,Género disco
0,2024,Album Of The Year,Charli xcx,BRAT,False,M,Dance Pop
1,2024,Album Of The Year,Jacob Collier,Djesse Vol. 4,False,H,Jazz
2,2024,Album Of The Year,Billie Eilish,HIT ME HARD AND SOFT,False,M,Pop
3,2024,Album Of The Year,Chappell Roan,The Rise And Fall Of A Midwest Princess,False,M,Pop
4,2024,Album Of The Year,Taylor Swift,THE TORTURED POETS DEPARTMENT,False,M,Pop


In [ ]:
import pandas as pd


billboard = pd.read_csv("BBD_Billboard_copia_Limpieza_2000.csv")
grammys = pd.read_csv("Album_del_Ano.csv")


grammys = grammys.rename(columns={
    'A�o': 'Año',
    'Categoría': 'Categoría',
    'Artista': 'Artista',
    'Disco o canción': 'Disco o canción',
    'Ganador': 'Ganador',
    'Género del artista/banda': 'Género del artista/banda',
    'Género disco': 'Género disco'
})


billboard = billboard.rename(columns={
    'artist': 'Artista',
    'peak_position': 'Peak',
    'song': 'Canción Billboard',
    'year': 'Año'
})


if "Año" not in billboard.columns:
    billboard["Año"] = pd.to_datetime(billboard["date"]).dt.year


def limpiar_texto(x):
    if isinstance(x, str):
        x = x.lower().strip()
        x = x.replace("&", "and")
        x = x.replace("feat.", "featuring").replace("feat", "featuring")
        x = x.replace("ft.", "featuring").replace("ft", "featuring")
        return x
    return x

billboard["Artista_norm"] = billboard["Artista"].apply(limpiar_texto)
grammys["Artista_norm"]   = grammys["Artista"].apply(limpiar_texto)


billboard_top10 = billboard[billboard["Peak"] <= 10]


union = billboard_top10.merge(
    grammys,
    on=["Año", "Artista_norm"],
    how="inner",
    suffixes=(" Billboard", " Grammy")
)


resultado = union[[
    "Año",
    "Artista Billboard",
    "Canción Billboard",
    "Peak"
]]

print(resultado)
resultado.head(20)

       Año  Artista Billboard     Canción Billboard  Peak
0     2000             Eminem   The Real Slim Shady     1
1     2000             Eminem   The Real Slim Shady     7
2     2000             Eminem   The Real Slim Shady     6
3     2000             Eminem   The Real Slim Shady     4
4     2000             Eminem   The Real Slim Shady     4
...    ...                ...                   ...   ...
3638  2024  Sabrina Carpenter  Please Please Please     1
3639  2024      Billie Eilish    Birds Of A Feather     2
3640  2024  Sabrina Carpenter                 Taste     2
3641  2024  Sabrina Carpenter              Espresso     3
3642  2024      Chappell Roan      Good Luck; Babe!     4

[3643 rows x 4 columns]


,Año,Artista Billboard,Canción Billboard,Peak
0,2000,Eminem,The Real Slim Shady,1
1,2000,Eminem,The Real Slim Shady,7
2,2000,Eminem,The Real Slim Shady,6
3,2000,Eminem,The Real Slim Shady,4
4,2000,Eminem,The Real Slim Shady,4
5,2000,Eminem,The Real Slim Shady,4
6,2000,Eminem,The Real Slim Shady,4
7,2000,Eminem,The Real Slim Shady,4
8,2000,Eminem,The Real Slim Shady,4
9,2000,Eminem,The Real Slim Shady,4


In [ ]:
!pip install Unidecode

import pandas as pd
import altair as alt
import unidecode
import numpy as np


album = pd.read_csv("Album_del_Ano.csv")
billboard = pd.read_csv("BBD_Billboard_copia_Limpieza_2000.csv")


album.columns = [
    "Año", "Categoria", "Artista", "Disco_o_cancion",
    "Ganador", "Genero_artista", "Genero_disco"
]


billboard["Año"] = pd.to_datetime(billboard["date"], errors="coerce").dt.year


album["Artista_norm"] = album["Artista"].apply(lambda x: unidecode.unidecode(str(x)).lower())
billboard["artist_norm"] = billboard["artist"].apply(lambda x: unidecode.unidecode(str(x)).lower())


artistas_fijos = [
    "Adele", "Billie Eilish", "Bruno Mars", "Harry Styles",
    "Mumford & Sons", "Norah Jones", "Outkast", "Taylor Swift", "U2"
]
artistas_fijos_norm = [unidecode.unidecode(a).lower() for a in artistas_fijos]


subset = billboard[billboard["artist_norm"].isin(artistas_fijos_norm)].copy()


ganadores = album[album["Ganador"] == 1][["Artista", "Año", "Disco_o_cancion"]].copy()
ganadores["Artista_norm"] = ganadores["Artista"].apply(lambda x: unidecode.unidecode(str(x)).lower())


años_ganados = (
    ganadores.groupby("Artista_norm")["Año"]
    .apply(lambda x: ", ".join(x.astype(str)))
    .to_dict()
)


subset["años_ganó_album_del_año"] = subset["artist_norm"].map(años_ganados)


np.random.seed(1)
subset["y_jitter"] = (
    subset.groupby("artist").cumcount() * 0.25
    + np.random.uniform(-0.1, 0.1, len(subset))
)


chart = (
    alt.Chart(subset)
    .mark_circle(filled=True)
    .encode(
        x=alt.X("Año:O", title="Año"),
        y=alt.Y("artist:N", title="Artista"),
        yOffset="y_jitter:Q",

        size=alt.Size(
            "peak_position:Q",
            scale=alt.Scale(domain=[1, 100], range=[2000, 20]),
            title="Peak Billboard (1 = mejor)"
        ),

        color=alt.Color(
            "artist:N",
            title="Artista",
            scale=alt.Scale(
                range=[
                    "#EAC119", "#808BC5", "#EAA7C7", "#9ED6DF", "#245E55",
                    "#ED773C", "#C63F3E", "#1D1D1B", "#D8639C", "#EDD470",
                    "#49B5A3", "#444E7D", "#24B2C9", "#DA7676", "#F1A781"
                ]
            )
        ),

        tooltip=[
            "artist:N",
            "Año:O",
            "peak_position:Q",
            alt.Tooltip("años_ganó_album_del_año:N", title="Ganó Álbum del Año en"),
        ]
    )
    .properties(
        width=850,
        height=550,
        title="Peaks en la lista Billboard de los ganadores a Álbum del Año"
    )
)

chart


alt.Chart(...)

In [ ]:
from google.colab import files
files.download('GráficosTop10.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

billboard = pd.read_csv("BBD_Billboard_copia_Limpieza_2000.csv")
grammys = pd.read_csv("Cancion_del_Ano.csv", encoding="latin1")

print(grammys.columns)

ParserError: Error tokenizing data. C error: Expected 7 fields in line 6, saw 8


In [ ]:
grammys = pd.read_csv("Cancion_del_Ano.csv", encoding="latin1", sep=';')
print(grammys.head())


  Aï¿½o,Categorï¿½a,Artista,Disco o canciï¿½n,Ganador,Gï¿½nero del artista/banda,Gï¿½nero canciï¿½n
0  2024,Song Of The Year,Shaboozey,A Bar Song (Ti...                                               
1  2024,Song Of The Year,Billie Eilish,BIRDS OF A...                                               
2  2024,Song Of The Year,Bruno Mars featuring Lad...                                               
3  2024,Song Of The Year,Taylor Swift,Fortnight,F...                                               
4  2024,Song Of The Year,Chappell Roan,Good Luck,...                                               


In [ ]:
import pandas as pd

# === LEER ARCHIVO CON CODIFICACIÓN CORRECTA ===
grammys = pd.read_csv("Cancion_del_Ano.csv", encoding="latin1", sep=",", on_bad_lines='skip')

print(grammys.columns)
grammys.head()

Index(['Aï¿½o', 'Categorï¿½a', 'Artista', 'Disco o canciï¿½n', 'Ganador',
       'Gï¿½nero del artista/banda', 'Gï¿½nero canciï¿½n'],
      dtype='object')


,Aï¿½o,Categorï¿½a,Artista,Disco o canciï¿½n,Ganador,Gï¿½nero del artista/banda,Gï¿½nero canciï¿½n
0,2024,Song Of The Year,Shaboozey,A Bar Song (Tipsy),False,H,Country
1,2024,Song Of The Year,Billie Eilish,BIRDS OF A FEATHER,False,M,Pop
2,2024,Song Of The Year,Bruno Mars featuring Lady Gaga,Die With A Smile,False,MIXTO,Pop
3,2024,Song Of The Year,Taylor Swift,Fortnight,False,M,Pop
4,2024,Song Of The Year,Sabrina Carpenter,Please Please Please,False,M,Pop


In [ ]:
import pandas as pd

# === CARGAR BASES ===

billboard = pd.read_csv("BBD_Billboard_copia_Limpieza_2000.csv")

grammys = pd.read_csv("Cancion_del_Ano.csv", encoding="latin1", on_bad_lines='skip')

grammys = grammys.rename(columns={
    'Aï¿½o': 'Año',
    'Categorï¿½a': 'Categoría',
    'Artista': 'Artista',
    'Disco o canciï¿½n': 'Disco o canción',
    'Ganador': 'Ganador',
    'Gï¿½nero del artista/banda': 'Género del artista/banda',
    'Gï¿½nero canciï¿½n': 'Género canción'
})

# === RENOMBRAR BILLBOARD ===
billboard = billboard.rename(columns={
    'artist': 'Artista',
    'peak_position': 'Peak',
    'song': 'Canción Billboard',
    'year': 'Año'
})

# Crear columna Año desde date si falta
if "Año" not in billboard.columns:
    billboard["Año"] = pd.to_datetime(billboard["date"]).dt.year

# === NORMALIZAR NOMBRES DE ARTISTAS ===
def limpiar_texto(x):
    if isinstance(x, str):
        x = x.lower().strip()
        x = x.replace("&", "and")
        x = x.replace("feat.", "featuring").replace("feat", "featuring")
        x = x.replace("ft.", "featuring").replace("ft", "featuring")
        return x
    return x

billboard["Artista_norm"] = billboard["Artista"].apply(limpiar_texto)
grammys["Artista_norm"]   = grammys["Artista"].apply(limpiar_texto)

# === FILTRO TOP 10 ===
billboard_top10 = billboard[billboard["Peak"] <= 10]

# === MERGE ===
union = billboard_top10.merge(
    grammys,
    on=["Año", "Artista_norm"],
    how="inner",
    suffixes=(" Billboard", " Grammy")
)

resultado = union[[
    "Año",
    "Artista Billboard",
    "Canción Billboard",
    "Peak"
]]

print(resultado)
resultado.head(20)


       Año  Artista Billboard    Canción Billboard  Peak
0     2000          Macy Gray                I Try     1
1     2000         Faith Hill              Breathe     5
2     2000         Faith Hill              Breathe     3
3     2000         Faith Hill              Breathe     3
4     2000         Faith Hill  The Way You Love Me     1
...    ...                ...                  ...   ...
3918  2024  Sabrina Carpenter             Espresso     3
3919  2024     Kendrick Lamar          Not Like Us     1
3920  2024     Kendrick Lamar    Wacced Out Murals     4
3921  2024     Kendrick Lamar         Reincarnated     8
3922  2024     Kendrick Lamar    Man At The Garden     9

[3923 rows x 4 columns]


,Año,Artista Billboard,Canción Billboard,Peak
0,2000,Macy Gray,I Try,1
1,2000,Faith Hill,Breathe,5
2,2000,Faith Hill,Breathe,3
3,2000,Faith Hill,Breathe,3
4,2000,Faith Hill,The Way You Love Me,1
5,2000,Destiny's Child,Say My Name,1
6,2000,Faith Hill,Breathe,3
7,2000,Destiny's Child,Say My Name,1
8,2000,Faith Hill,Breathe,3
9,2000,Destiny's Child,Say My Name,1


In [ ]:
!pip install Unidecode

import pandas as pd
import altair as alt
import unidecode
import numpy as np


# === 1. Cargar bases ===
cancion = pd.read_csv("Cancion_del_Ano.csv", encoding="latin1", on_bad_lines='skip')
billboard = pd.read_csv("BBD_Billboard_copia_Limpieza_2000.csv")


# === 2. Renombrar columnas con nombres consistentes ===
# (Adaptado a tus columnas reales del CSV)
cancion.columns = [
    "Año", "Categoria", "Artista", "Disco_o_cancion",
    "Ganador", "Genero_artista", "Genero_cancion"
]


# === 3. Crear columna con año en Billboard ===
billboard["Año"] = pd.to_datetime(billboard["date"], errors="coerce").dt.year


# === 4. Normalización de artistas ===
cancion["Artista_norm"] = cancion["Artista"].apply(lambda x: unidecode.unidecode(str(x)).lower())
billboard["artist_norm"] = billboard["artist"].apply(lambda x: unidecode.unidecode(str(x)).lower())


# === 5. Lista de artistas a resaltar ===
artistas_fijos = [
    "Adele", "Billie Eilish", "Bruno Mars", "Harry Styles",
    "Mumford & Sons", "Norah Jones", "Outkast", "Taylor Swift", "U2"
]
artistas_fijos_norm = [unidecode.unidecode(a).lower() for a in artistas_fijos]


subset = billboard[billboard["artist_norm"].isin(artistas_fijos_norm)].copy()


# === 6. Filtrar ganadores de Canción del Año ===
ganadores_cancion = cancion[cancion["Ganador"] == 1][["Artista", "Año", "Disco_o_cancion"]].copy()
ganadores_cancion["Artista_norm"] = ganadores_cancion["Artista"].apply(lambda x: unidecode.unidecode(str(x)).lower())


# === 7. Mapear años ganados (pueden ser varios) ===
años_ganados_cancion = (
    ganadores_cancion.groupby("Artista_norm")["Año"]
    .apply(lambda x: ", ".join(x.astype(str)))
    .to_dict()
)

subset["años_ganó_cancion_del_año"] = subset["artist_norm"].map(años_ganados_cancion)


# === 8. Jitter para separar puntos dentro de un mismo año ===
np.random.seed(1)
subset["y_jitter"] = (
    subset.groupby("artist").cumcount() * 0.25
    + np.random.uniform(-0.1, 0.1, len(subset))
)


# === 9. Gráfico ===
chart = (
    alt.Chart(subset)
    .mark_circle(filled=True)
    .encode(
        x=alt.X("Año:O", title="Año"),
        y=alt.Y("artist:N", title="Artista"),
        yOffset="y_jitter:Q",

        size=alt.Size(
            "peak_position:Q",
            scale=alt.Scale(domain=[1, 100], range=[2000, 20]),
            title="Peak Billboard (1 = mejor)"
        ),

        color=alt.Color(
            "artist:N",
            title="Artista",
            scale=alt.Scale(
                range=[
                    "#EAC119", "#808BC5", "#EAA7C7", "#9ED6DF", "#245E55",
                    "#ED773C", "#C63F3E", "#1D1D1B", "#D8639C", "#EDD470",
                    "#49B5A3", "#444E7D", "#24B2C9", "#DA7676", "#F1A781"
                ]
            )
        ),

        tooltip=[
            "artist:N",
            "Año:O",
            "peak_position:Q",
            alt.Tooltip("años_ganó_cancion_del_año:N", title="Ganó Canción del Año en"),
        ]
    )
    .properties(
        width=850,
        height=550,
        title="Peaks en Billboard de los ganadores a Canción del Año"
    )
)

chart

alt.Chart(...)

In [ ]:
!pip install Unidecode

import pandas as pd
import altair as alt
import unidecode
import numpy as np


# === 1. Cargar bases ===
cancion = pd.read_csv("Cancion_del_Ano.csv", encoding="latin1", on_bad_lines='skip')
billboard = pd.read_csv("BasedeDatos_BBD_Billboard_copia_Limpieza_2000.csv")


# === 2. Renombrar columnas ===
cancion.columns = [
    "Año", "Categoria", "Artista", "Disco_o_cancion",
    "Ganador", "Genero_artista", "Genero_cancion"
]


# === 3. Año Billboard ===
billboard["Año"] = pd.to_datetime(billboard["date"], errors="coerce").dt.year


# === 4. Normalizar artistas ===
cancion["Artista_norm"] = cancion["Artista"].apply(lambda x: unidecode.unidecode(str(x)).lower())
billboard["artist_norm"] = billboard["artist"].apply(lambda x: unidecode.unidecode(str(x)).lower())


# === 5. Filtrar ganadores de Canción del Año ===
ganadores = cancion[cancion["Ganador"] == 1].copy()

ganadores["Artista_norm"] = ganadores["Artista"].apply(
    lambda x: unidecode.unidecode(str(x)).lower()
)


# === 6. Detectar ganadores que también aparecen en Billboard ===
ganadores_en_billboard = ganadores[
    ganadores["Artista_norm"].isin(billboard["artist_norm"])
].copy()

lista_ganadores_norm = ganadores_en_billboard["Artista_norm"].unique()


# === 7. Subset Billboard con esos artistas ===
subset = billboard[billboard["artist_norm"].isin(lista_ganadores_norm)].copy()


# === 8. Agregar años en que ganaron Canción del Año ===
años_por_artista = (
    ganadores_en_billboard.groupby("Artista_norm")["Año"]
    .apply(lambda x: ", ".join(x.astype(str)))
    .to_dict()
)

subset["años_ganó_cancion_del_año"] = subset["artist_norm"].map(años_por_artista)


# === 9. Jitter ===
np.random.seed(1)
subset["y_jitter"] = (
    subset.groupby("artist").cumcount() * 0.25
    + np.random.uniform(-0.1, 0.1, len(subset))
)


# === 10. Gráfico ===
chart = (
    alt.Chart(subset)
    .mark_circle(filled=True)
    .encode(
        x=alt.X("Año:O", title="Año"),
        y=alt.Y("artist:N", title="Artista"),
        yOffset="y_jitter:Q",

        size=alt.Size(
            "peak_position:Q",
            scale=alt.Scale(domain=[1, 100], range=[2000, 20]),
            title="Peak Billboard (1 = mejor)"
        ),
    color=alt.Color(
            "artist:N",
            title="Artista",
            scale=alt.Scale(
                range=[
                    "#EAC119", "#808BC5", "#EAA7C7", "#9ED6DF", "#245E55",
                    "#ED773C", "#C63F3E", "#1D1D1B", "#D8639C", "#EDD470",
                    "#49B5A3", "#444E7D", "#24B2C9", "#DA7676", "#F1A781"
                ]
            )
        ),
    )
    .properties(
        width=850,
        height=550,
        title="Ganadores de Canción del Año presentes en Billboard"
    )
)

chart

alt.Chart(...)

In [ ]:
import pandas as pd
import altair as alt
import unidecode
import numpy as np

# === 1. Cargar bases ===
cancion = pd.read_csv("Cancion_del_Ano.csv", encoding="latin1", on_bad_lines='skip')
billboard = pd.read_csv("BasedeDatos_BBD_Billboard_copia_Limpieza_2000.csv")

# === 2. Renombrar columnas ===
cancion.columns = [
    "Año", "Categoria", "Artista", "Disco_o_cancion",
    "Ganador", "Genero_artista", "Genero_cancion"
]

# === 3. Año Billboard ===
billboard["Año"] = pd.to_datetime(billboard["date"], errors="coerce").dt.year

# === 4. Normalizar artistas ===
cancion["Artista_norm"] = cancion["Artista"].apply(lambda x: unidecode.unidecode(str(x)).lower())
billboard["artist_norm"] = billboard["artist"].apply(lambda x: unidecode.unidecode(str(x)).lower())

# === 5. Filtrar solo ganadores Canción del Año ===
ganadores = cancion[cancion["Ganador"] == 1].copy()
ganadores["Artista_norm"] = ganadores["Artista"].apply(lambda x: unidecode.unidecode(str(x)).lower())

# === 6. Ganadores que aparecen en Billboard ===
ganadores_en_billboard = ganadores[
    ganadores["Artista_norm"].isin(billboard["artist_norm"])
].copy()

lista_ganadores_norm = ganadores_en_billboard["Artista_norm"].unique()

# === 7. Subset Billboard con esos artistas ===
subset = billboard[billboard["artist_norm"].isin(lista_ganadores_norm)].copy()

# === 8. Años en que ganaron Canción del Año ===
años_por_artista = (
    ganadores_en_billboard.groupby("Artista_norm")["Año"]
    .apply(lambda x: ", ".join(x.astype(str)))
    .to_dict()
)

subset["años_ganó_cancion_del_año"] = subset["artist_norm"].map(años_por_artista)

# === 9. Jitter para separar puntos ===
np.random.seed(1)
subset["y_jitter"] = (
    subset.groupby("artist").cumcount() * 0.25
    + np.random.uniform(-0.1, 0.1, len(subset))
)

# === 10. Gráfico ===
chart = (
    alt.Chart(subset)
    .mark_circle(filled=True)
    .encode(
        x=alt.X("Año:O", title="Año"),
        y=alt.Y("artist:N", title="Artista"),
        yOffset="y_jitter:Q",

        size=alt.Size(
            "peak_position:Q",
            scale=alt.Scale(domain=[1, 100], range=[2000, 20]),
            title="Peak Billboard (1 = mejor)"
        ),

        color=alt.Color(
            "artist:N",
            title="Artista",
            scale=alt.Scale(
                range=[
                    "#EAC119", "#808BC5", "#EAA7C7", "#9ED6DF", "#245E55",
                    "#ED773C", "#C63F3E", "#1D1D1B", "#D8639C", "#EDD470",
                    "#49B5A3", "#444E7D", "#24B2C9", "#DA7676", "#F1A781"
                ]
            )
        ),

        tooltip=[
            alt.Tooltip("artist:N", title="Artista"),
            alt.Tooltip("Año:O", title="Año en Billboard"),
            alt.Tooltip("peak_position:Q", title="Peak"),
            alt.Tooltip("años_ganó_cancion_del_año:N", title="Ganó Canción del Año en")
        ]
    )
    .properties(
        width=850,
        height=550,
        title="Ganadores de Canción del Año presentes en Billboard"
    )
)

chart



alt.Chart(...)

In [ ]:
import pandas as pd
import altair as alt

# === 1. Prepare Grammy data for Taylor Swift ===
# Use the existing 'ts_grammy' DataFrame (from previous cells)

# Convert the 'Ganó' column to text for plotting
ts_grammy["Resultado"] = ts_grammy["Ganó"].apply(lambda x: "Ganó" if x else "Nominada")

# === 2. Group by year and result: count nominations and wins ===
ts_resumen = (
    ts_grammy.groupby(["Año", "Resultado"])
    .size()
    .reset_index(name="Cantidad")
)

# === 3. Create a line chart ===
line_chart = (
    alt.Chart(ts_resumen)
    .mark_line(point=True, strokeWidth=3)
    .encode(
        x=alt.X("Año:O", title="Año"),
        y=alt.Y("Cantidad:Q", title="Número de nominaciones/ganadas"),
        color=alt.Color(
            "Resultado:N",
            scale=alt.Scale(range=["#808BC5", "#EAC119"]),  # azul = nominada, amarillo = ganó
            title="Resultado"
        ),
        tooltip=["Año", "Resultado", "Cantidad"]
    )
    .properties(
        width=800,
        height=400,
        title="Taylor Swift – Nominaciones y Victorias Grammy por Año"
    )
)

line_chart

TypeError: read_csv() takes 1 positional argument but 4 were given

In [79]:
import altair as alt
import pandas as pd

# --- CARGA DE DATOS ---
ts = pd.read_csv("Taylor_Billboard.csv", encoding="latin1")
grammy = pd.read_csv("Album_del_Ano.csv", encoding="latin1")

grammy = grammy.rename(columns={
    'Aï¿½o': 'Año',
    'Disco o canciï¿½n': 'Disco o canción',
    'Categorï¿½a': 'Categoría' # Added this line to rename the column
})

# --- PROCESAR DATOS ---
ts["Fecha"] = pd.to_datetime(ts["Fecha"], errors="coerce", format='%Y-%m-%d')
ts["Año"] = ts["Fecha"].dt.year
ts["Peak 1-10"] = pd.to_numeric(ts["Peak 1-10"], errors="coerce")

metric = (
    ts[ts["Peak 1-10"] <= 10]
    .groupby("Año")
    .size()
    .reset_index(name="Hits_Top10")
)

grammy_taylor = grammy[grammy["Artista"].str.contains("Taylor Swift", case=False)]

df_grammys = pd.DataFrame({
    "Año": grammy_taylor["Año"],
    "Evento": grammy_taylor["Categoría"],
    "Ganó": grammy_taylor["Ganador"].apply(lambda x: "Ganó" if x == 1 else "Nominada")
})

# Posición superior para mostrar los íconos
y_max = metric["Hits_Top10"].max() + 1.5

# --- LÍNEA PRINCIPAL ---
line = (
    alt.Chart(metric)
    .mark_line(color="#808BC5", strokeWidth=3)
    .encode(
        x=alt.X("Año:O", title="Año"),
        y=alt.Y("Hits_Top10:Q", title="Canciones en Top 10")
    )
    .properties(width=950, height=480)
)

# --- ÍCONOS (♪) ESTILO S&P500 ---
grammy_icons = (
    alt.Chart(df_grammys)
    .mark_text(
        text="♪",
        fontSize=40,
        fontWeight="bold",
        color="gold"
    )
    .encode(
        x="Año:O",
        y=alt.value(y_max),
        color=alt.Color(
            "Ganó:N",
            scale=alt.Scale(
                domain=["Ganó", "Nominada"],
                range=["#EAC119", "#C39BD3"]
            ),
            title="Resultado Grammy"
        ),
        tooltip=["Año", "Evento", "Ganó"]
    )
)

# --- CAJAS CON BORDE ALREDEDOR DEL TEXTO ---
grammy_boxes = (
    alt.Chart(df_grammys)
    .mark_text(
        align="center",
        baseline="top",
        dy=40,              # separación vertical debajo del ícono
        fontSize=11,
        fontWeight="bold",
        color="black",
        stroke="black",
        strokeWidth=0.2
    )
    .encode(
        x="Año:O",
        y=alt.value(y_max),
        text="Evento"
    )
)

# --- FLECHAS ESTILO S&P 500 ---
arrows = (
    alt.Chart(df_grammys)
    .mark_rule(color="gray", strokeWidth=1)
    .encode(
        x="Año:O",
        y=alt.value(y_max - 0.3),
        y2=alt.value(y_max - 1.2)
    )
)

arrow_heads = (
    alt.Chart(df_grammys)
    .mark_point(shape="triangle-down", size=80, color="gray")
    .encode(
        x="Año:O",
        y=alt.value(y_max - 1.2)
    )
)

# --- TÍTULO SEPARADO Y LIMPIO ---
title = (
    alt.Chart()
    .mark_text(
        text="Taylor Swift: Evolución en Billboard y Años de Grammy (AOTY)",
        fontSize=22,
        fontWeight="bold",
        align="center",
        dy=-240
    )
    .encode()
)

subtitle = (
    alt.Chart()
    .mark_text(
        text="Línea = Canciones Top 10 por año | Íconos ♪ = AOTY (dorado=ganó, púrpura=nominada)",
        fontSize=14,
        color="gray",
        align="center",
        dy=-215
    )
)

# --- ENSAMBLAR TODO ---
final_chart = (
      line
    + grammy_icons
    + grammy_boxes
    + arrows
    + arrow_heads
    + title
    + subtitle
)

final_chart

alt.LayerChart(...)

In [ ]:
import altair as alt
import pandas as pd

# --- CARGA DE DATOS ---
bb = pd.read_csv("MAROON5_BILLBOARD.csv", encoding="latin1")
grammy = pd.read_csv("MAROON5_GRAMMY.csv", encoding="latin1")

# --- NORMALIZAR NOMBRES DE COLUMNAS (por si vienen con errores)
grammy = grammy.rename(columns={
    'Aï¿½o': 'Año',
    'Disco o canciï¿½n': 'Disco o canción',
    'Disco o cancion': 'Disco o canción'
})

# --- PROCESAR DATOS ---
bb["Fecha"] = pd.to_datetime(bb["Fecha"], errors="coerce")
bb["Año"] = bb["Fecha"].dt.year
bb["Peak 1-5"] = pd.to_numeric(bb["Peak 1-5"], errors="coerce")

# hits top 10 por año
metric = (
    bb[bb["Peak 1-5"] <= 10]
    .groupby("Año")
    .size()
    .reset_index(name="Hits_Top5")
)

# dataframe de nominaciones/ganadas
df_grammys = pd.DataFrame({
    "Año": grammy["Año"],
    "Evento": grammy["Categoria"],
    "Ganó": grammy["Ganador"].apply(lambda x: "Ganó" if x == 1 else "Nominada")
})

# posición superior para los íconos
y_max = metric["Hits_Top5"].max() + 1.5

# --- LÍNEA PRINCIPAL ---
line = (
    alt.Chart(metric)
    .mark_line(color="#E63946", strokeWidth=3)  # rojo Maroon 5 vibes
    .encode(
        x=alt.X("Año:O", title="Año"),
        y=alt.Y("Hits_Top5:Q", title="Canciones en Top 5")
    )
    .properties(width=950, height=480)
)

# --- ÍCONOS (♪) ---
grammy_icons = (
    alt.Chart(df_grammys)
    .mark_text(
        text="♪",
        fontSize=40,
        fontWeight="bold"
    )
    .encode(
        x="Año:O",
        y=alt.value(y_max),
        color=alt.Color(
            "Ganó:N",
            scale=alt.Scale(
                domain=["Ganó", "Nominada"],
                range=["#EAC119", "#C39BD3"]
            ),
            title="Resultado Grammy"
        ),
        tooltip=["Año", "Evento", "Ganó"]
    )
)

# --- CAJAS CON BORDE ---
grammy_boxes = (
    alt.Chart(df_grammys)
    .mark_text(
        align="center",
        baseline="top",
        dy=40,
        fontSize=11,
        fontWeight="bold",
        color="black",
        stroke="black",
        strokeWidth=0.2
    )
    .encode(
        x="Año:O",
        y=alt.value(y_max),
        text="Evento"
    )
)

# --- FLECHAS ---
arrows = (
    alt.Chart(df_grammys)
    .mark_rule(color="gray", strokeWidth=1)
    .encode(
        x="Año:O",
        y=alt.value(y_max - 0.3),
        y2=alt.value(y_max - 1.2)
    )
)

arrow_heads = (
    alt.Chart(df_grammys)
    .mark_point(shape="triangle-down", size=80, color="gray")
    .encode(
        x="Año:O",
        y=alt.value(y_max - 1.2)
    )
)

# --- TÍTULO Y SUBTÍTULO ---
title = (
    alt.Chart()
    .mark_text(
        text="Maroon 5: Evolución en Billboard y Años de Grammy (AOTY)",
        fontSize=22,
        fontWeight="bold",
        align="center",
        dy=-240
    )
)

subtitle = (
    alt.Chart()
    .mark_text(
        text="Línea = Canciones Top 5 por año | Íconos ♪ = AOTY (dorado=ganó, púrpura=nominada)",
        fontSize=14,
        color="gray",
        align="center",
        dy=-215
    )
)

# --- ENSAMBLE ---
final_chart = (
      line
    + grammy_icons
    + grammy_boxes
    + arrows
    + arrow_heads
    + title
    + subtitle
)

final_chart
from google.colab import files

# Guardar el gráfico en un archivo HTML
final_chart.save("Maroon5_Billboard_Grammy.html")

# Descargar el archivo a tu computadora
files.download("Maroon5_Billboard_Grammy.html")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import altair as alt
import pandas as pd

# --- DATOS ---
billie = pd.read_csv("BILLIE_BILLBOARD.csv", encoding="latin1")
grammy = pd.read_csv("BILLIE_GRAMMY.csv", encoding="latin1")

# Rename the 'Disco o cancion' column to 'Disco o canción' in the grammy DataFrame
grammy = grammy.rename(columns={'Disco o cancion': 'Disco o canción'})

# --- PROCESAMIENTO Billboard ---
billie["Fecha"] = pd.to_datetime(billie["Fecha"], errors="coerce")
billie["Año"] = billie["Fecha"].dt.year
billie["Peak 1-5"] = pd.to_numeric(billie["Peak 1-5"], errors="coerce")

metric = (
    billie[billie["Peak 1-5"] <= 10]
    .groupby("Año")
    .size()
    .reset_index(name="Hits_Top10")
)

# --- Preparar datos Grammy (Sólo Billie + Song of the Year) ---
df_grammys = grammy.copy()
df_grammys = df_grammys[df_grammys["Artista"].str.contains("Billie Eilish", case=False)]

# Create a new column 'Resultado_Ganador' for the mapped values
df_grammys['Resultado_Ganador'] = df_grammys["Ganador"].map({True: "Ganó", False: "Nominada"})

# Posición vertical para colocar los íconos
y_max = metric["Hits_Top10"].max() + 1.5

# --- Gráfico ---
line = alt.Chart(metric).mark_line(color="#6AC4A6", strokeWidth=3).encode(
    x=alt.X("Año:O", title="Año"),
    y=alt.Y("Hits_Top10:Q", title="Canciones en Top 5")
).properties(width=950, height=480)

grammy_icons = alt.Chart(df_grammys).mark_text(
    text="♪", fontSize=40, fontWeight="bold"
).encode(
    x="Año:O",
    y=alt.value(y_max),
    color=alt.Color(
        "Resultado_Ganador:N", # Reference the new column by name
        scale=alt.Scale(domain=["Ganó", "Nominada"], range=["#EAC119", "#9AD0EC"]),
        title="Resultado – Grammy SOTY"
    ),
    tooltip=["Año", "Disco o canción", "Ganador"]
)

grammy_boxes = alt.Chart(df_grammys).mark_text(
    align="center", baseline="top", dy=40, fontSize=11, fontWeight="bold",
    color="black", stroke="black", strokeWidth=0.2
).encode(
    x="Año:O",
    y=alt.value(y_max),
    text="Disco o canción"
)

arrows = alt.Chart(df_grammys).mark_rule(color="gray", strokeWidth=1).encode(
    x="Año:O",
    y=alt.value(y_max - 0.3),
    y2=alt.value(y_max - 1.2)
)

arrow_heads = alt.Chart(df_grammys).mark_point(
    shape="triangle-down", size=80, color="gray"
).encode(
    x="Año:O",
    y=alt.value(y_max - 1.2)
)

title = alt.Chart().mark_text(
    text="Billie Eilish: Billboard Top 5 & Grammys – Canción del Año",
    fontSize=22, fontWeight="bold", align="center", dy=-240
)

subtitle = alt.Chart().mark_text(
    text="Línea = canciones Top 5 por año | Ícono ♪ = Grammy SOTY (oro=ganó, azul=nominada)",
    fontSize=14, color="gray", align="center", dy=-215
)

final_chart = (line + grammy_icons + grammy_boxes + arrows + arrow_heads + title + subtitle)
final_chart

from google.colab import files

# Guardar el gráfico en un archivo HTML
final_chart.save("Billie_Billboard_Grammy.html")

# Descargar el archivo a tu computadora
files.download("Billie_Billboard_Grammy.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import altair as alt
import pandas as pd

# --- Cargar datos ---
df = pd.read_csv("GENERO_MUSICAL.csv", encoding="latin1")

# Convertir año a número
df["Año"] = pd.to_numeric(df["Año"], errors="coerce")

# Convertir "Ganador" a etiquetas
df["Resultado"] = df["Ganador"].apply(lambda x: "Ganó" if x == True or x == 1 or x == "True" else "Nominada")

# Símbolos
df["Simbolo"] = df["Resultado"].map({
    "Ganó": "circle",
    "Nominada": "circle"
})

# Tamaños
df["Size"] = df["Resultado"].map({
    "Ganó": 300,
    "Nominada": 150
})

# NUEVOS COLORES:
df["Color"] = df["Resultado"].map({
    "Ganó": "#FFD800",      # Amarillo
    "Nominada": "#FF7EB9"   # Rosado
})

# --- Gráfico interactivo ---
timeline = (
    alt.Chart(df)
    .mark_point(filled=True)
    .encode(
        x=alt.X("Año:O", title="Año"),
        y=alt.value(0),
        shape=alt.Shape("Resultado:N", legend=None),
        size=alt.Size("Size:Q", legend=None),
        color=alt.Color("Color:N", legend=None),
        tooltip=["Año", "Artista", "Categoria", "Genero cancion", "Resultado"]
    )
    .properties(width=900, height=120)
)

# Texto encima del punto
labels = (
    alt.Chart(df)
    .mark_text(dy=-15, fontSize=12, fontWeight="bold")
    .encode(
        x="Año:O",
        y=alt.value(0),
        text="Artista"
    )
)

final_chart = timeline + labels
final_chart

from google.colab import files

# Guardar el gráfico en un archivo HTML
final_chart.save("GENERO_MUSICAL.html")

# Descargar el archivo a tu computadora
files.download("GENERO_MUSICAL.html")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import altair as alt
import pandas as pd

# ==========================
#   CARGA DE DATOS
# ==========================
bill = pd.read_csv("DIXIECHICKS_BILLBOARD.csv", encoding="latin1")
gram = pd.read_csv("DIXIECHICKS_GRAMMY.csv", encoding="latin1")

# ==========================
#   PROCESAMIENTO BILLBOARD
# ==========================
bill["Fecha"] = pd.to_datetime(bill["Fecha"], errors="coerce")
bill["Año"] = bill["Fecha"].dt.year
bill["Peak 1-5"] = pd.to_numeric(bill["Peak 1-5"], errors="coerce")

metric = (
    bill[bill["Peak 1-5"] <= 10]
    .groupby("Año")
    .size()
    .reset_index(name="Hits_Top5")
)

# ==========================
#   PROCESAMIENTO GRAMMY
# ==========================

# Normalizar valores de “Ganador”
gram["Ganador_norm"] = gram["Ganador"].astype(str).str.strip().str.lower().map({
    "true": True, "1": True, "sí": True, "si": True, "ganó": True, "gano": True,
    "false": False, "0": False, "no": False
})

gram["Resultado_Ganador"] = gram["Ganador_norm"].apply(
    lambda x: "Ganó" if x else "Nominada"
)

# Asegurar que aparece AOTY de 2006
gram = gram[gram["Año"] == 2006]

# Título que quieres mostrar sobre el ícono
gram["Premio"] = "Record of the Year"

# ==========================
#   CONSTRUIR GRÁFICO
# ==========================

y_max = metric["Hits_Top5"].max() + 10

line = alt.Chart(metric).mark_line(
    color="pink", strokeWidth=3
).encode(
    x=alt.X("Año:O", title="Año"),
    y=alt.Y(
        "Hits_Top5:Q",
        title="Posiciones en el Top 5",
        scale=alt.Scale(reverse=True),   # ← Invierte la escala (1 arriba → 4 abajo)
        axis=alt.Axis(values=[1, 2, 3, 4])       # ← Muestra solo enteros
    )
)

icons = alt.Chart(gram).mark_text(
    text="♪", fontSize=38, fontWeight="bold",  dy=160
).encode(
    x="Año:O",
    y=alt.value(y_max),
    color=alt.Color(
        "Resultado_Ganador:N",
        scale=alt.Scale(
            domain=["Ganó"],
            range=["#EAC119"]
        ),
        title="Resultado Grammy"
    ),
    tooltip=["Año", "Premio", "Resultado_Ganador"]
)

labels = alt.Chart(gram).mark_text(
    dy=180, fontSize=13, fontWeight="bold"
).encode(
    x="Año:O",
    y=alt.value(y_max),
    text="Premio"
)

title = alt.Chart().mark_text(
    text="Dixie Chicks: Evolución en Billboard y Años de Grammy (ROTY)",
    fontSize=30, fontWeight="bold",
    align="center",
    dy=-250
)

subtitle = alt.Chart().mark_text(
    text="Línea = Posiciones en el Top 5 por año | Íconos ♪ = ROTY",
    fontSize=15, color="gray",
    align="center",
    dy=-225
)

final = line + icons + labels + title + subtitle
final = final.properties(
    width=1400,     # ancho amplio
    height=600      # alto estilo dashboard
).configure_view(
    continuousWidth=1400,
    continuousHeight=600
).configure(
    autosize=alt.AutoSizeParams(
        type='pad',   # ← ESTA ES LA SOLUCIÓN
        contains='padding'
    )
)
from google.colab import files

# Guardar el gráfico en un archivo HTML
final.save("DIXIE_CHICKS_GRAFICO.html")

# Descargar el archivo a tu computadora
files.download("DIXIE_CHICKS_GRAFICO.html")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>